# 작업형2 기출 유형(심화)
- 본 문제는 변형한 심화 문제 입니다.

### 여행 보험 패키지 상품을 구매할 확률 값을 구하시오
- 예측할 값(y): TravelInsurance (여행보험 패지지를 구매 했는지 여부 0:구매안함, 1:구매)
- 평가: roc-auc 평가지표
- data: t2-1-train.csv, t2-1-test.csv
- 제출 형식

~~~
id,TravelInsurance
0,0.3
1,0.48
2,0.3
3,0.83
~~~

# Baseline
### 3회 기출문제에서 데이터 셋을 편집해 조금 더 어렵게 만들었어요
- 결측치 추가
- Employment Type 컬럼에 카테고리 추가 
- sample_submission 파일은 제공된 적 없음(3회 때 제출 형식에 대한 이슈가 있어 제공하거나 제출 형식을 명확하게 설명할 가능성 있어 보임)

### 풀이 영상: https://youtu.be/QpNufh_ZV7A?t=291

In [ ]:
# 라이브러리 불러오기
import pandas as pd

In [ ]:
# 데이터 불러오기
train = pd.read_csv("../input/big-data-analytics-certification/t2-1-train.csv")
test = pd.read_csv("../input/big-data-analytics-certification/t2-1-test.csv")

# EDA

In [ ]:
# 데이터 사이즈
train.shape, train.shape

In [ ]:
# 샘플 확인
train.head()

In [ ]:
# type 확인
train.info()

실제 시험에서는 train과 test 카테고리가 같았어요. 만약 test데이터에 새로운 카테고리가 있다면 어떻게 풀어야 할까요?

In [ ]:
# 카테고리 수 확인
train.describe(include="object")

In [ ]:
# 카테고리 수 확인
test.describe(include="object")

In [ ]:
# Employment Type 컬럼 카테고리
train['Employment Type'].value_counts()

In [ ]:
# Employment Type 컬럼 카테고리
test['Employment Type'].value_counts()

In [ ]:
# 수치형 통계 값
train.describe(exclude="object")

In [ ]:
# 수치형 통계 값
test.describe(exclude="object")

실제 시험에서는 결측치가 없었어요. 만약 결측치가 있다면 어떻게 풀어야 할까요?

In [ ]:
# 결측치 확인
train.isnull().sum()

In [ ]:
# 결측치 확인
test.isnull().sum()

In [ ]:
# target
train['TravelInsurance'].value_counts()

# Data pre-processing

In [ ]:
# 결측치 처리
train['AnnualIncome'] = train['AnnualIncome'].fillna(train['AnnualIncome'].mean())
test['AnnualIncome'] = test['AnnualIncome'].fillna(test['AnnualIncome'].mean())

In [ ]:
# target값 변수에 옮기기
target = train.pop('TravelInsurance')

In [ ]:
# 데이터 합치기
df = pd.concat([train, test])
df.shape

In [ ]:
# 레이블 인코딩
from sklearn.preprocessing import LabelEncoder

cols = df.select_dtypes(include="object").columns
le = LabelEncoder()

for col in cols:
    df[col] = le.fit_transform(df[col])

In [ ]:
# train test 다시 분리
train = df[:train.shape[0]].copy()
test = df[train.shape[0]:].copy()

In [ ]:
# 스케일
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

train['AnnualIncome'] = scaler.fit_transform(train[['AnnualIncome']])
test['AnnualIncome'] = scaler.transform(test[['AnnualIncome']])

# 검증 데이터 분리

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(train, target, test_size=0.2, random_state=2022)
X_train.shape, X_val.shape, y_train.shape, y_val.shape

# 모델 학습 및 평가

In [ ]:
# 의사결정나무
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(random_state=2022)
model.fit(X_train, y_train)
pred = model.predict_proba(X_val)

In [ ]:
# 평가
from sklearn.metrics import roc_auc_score
roc_auc_score(y_val, pred[:,1])

In [ ]:
# 랜덤포레스트
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(random_state=2022)
model.fit(X_train, y_train)
pred = model.predict_proba(X_val)
roc_auc_score(y_val, pred[:,1])

In [ ]:
# xgboost
import xgboost as xgb
model = xgb.XGBRFClassifier(random_state=2022)
model.fit(X_train, y_train)
pred = model.predict_proba(X_val)
roc_auc_score(y_val, pred[:,1])

# 예측

In [ ]:
# test 데이터 예측
model = RandomForestClassifier(random_state=2022)
model.fit(X_train, y_train)
pred = model.predict_proba(test)

In [ ]:
# 예측한 데이터 -> 데이터프레임으로
submit = pd.DataFrame()
submit['id'] = test['id']
submit['TravelInsurance'] = pred[:,1]

In [ ]:
# 예측한 데이터 확인
submit.head()

In [ ]:
# csv 저장
submit.to_csv("2022.csv", index=False)

In [ ]:
# csv 확인
pd.read_csv("2022.csv")

# 추가

In [ ]:
# 만약 sample_submission이 주어진다면
sample_submission = pd.read_csv("../input/big-data-analytics-certification/t2-1-sample_submission.csv")
sample_submission['TravelInsurance'] = pred[:,1]
sample_submission.to_csv("2022.csv",index=False)